In [3]:
from pathlib import Path
from collections import defaultdict, Counter
import polars as pl

In [4]:
#1.path
CURRENT_DIR = Path.cwd()

PROJECT_ROOT = CURRENT_DIR.parent

DATA_ROOT = PROJECT_ROOT / "tennis_data"

EXTRACT_ROOT = DATA_ROOT / "extracted"


print("PROJECT ROOT:", PROJECT_ROOT)
print("DATA ROOT:", DATA_ROOT)
print("EXTRACT ROOT:", EXTRACT_ROOT)

PROJECT ROOT: /Users/macbook/Desktop/ data_analysis_tannis projects/tennis_data_analysis
DATA ROOT: /Users/macbook/Desktop/ data_analysis_tannis projects/tennis_data_analysis/tennis_data
EXTRACT ROOT: /Users/macbook/Desktop/ data_analysis_tannis projects/tennis_data_analysis/tennis_data/extracted


In [5]:
# 2. Find all Time Parquet files

time_files = sorted(
    EXTRACT_ROOT.glob(
        "*/time_*.parquet"
    )
)


print("Number of Time files:",len(time_files))


Number of Time files: 35671


In [6]:
# 3. Display some Time files


for file in time_files[:10]:
    print(file)

/Users/macbook/Desktop/ data_analysis_tannis projects/tennis_data_analysis/tennis_data/extracted/20240201/time_11974053.parquet
/Users/macbook/Desktop/ data_analysis_tannis projects/tennis_data_analysis/tennis_data/extracted/20240201/time_11974066.parquet
/Users/macbook/Desktop/ data_analysis_tannis projects/tennis_data_analysis/tennis_data/extracted/20240201/time_11998445.parquet
/Users/macbook/Desktop/ data_analysis_tannis projects/tennis_data_analysis/tennis_data/extracted/20240201/time_11998446.parquet
/Users/macbook/Desktop/ data_analysis_tannis projects/tennis_data_analysis/tennis_data/extracted/20240201/time_11998447.parquet
/Users/macbook/Desktop/ data_analysis_tannis projects/tennis_data_analysis/tennis_data/extracted/20240201/time_11998448.parquet
/Users/macbook/Desktop/ data_analysis_tannis projects/tennis_data_analysis/tennis_data/extracted/20240201/time_11998449.parquet
/Users/macbook/Desktop/ data_analysis_tannis projects/tennis_data_analysis/tennis_data/extracted/2024020

In [7]:
# 4. Check one Time file


test_time_df = pl.read_parquet(
    time_files[0]
)


print(test_time_df)


print("\nSchema:")
print(test_time_df.schema)

shape: (1, 7)
┌──────────┬──────────┬──────────┬──────────┬──────────┬──────────┬────────────────────────────────┐
│ match_id ┆ period_1 ┆ period_2 ┆ period_3 ┆ period_4 ┆ period_5 ┆ current_period_start_timestamp │
│ ---      ┆ ---      ┆ ---      ┆ ---      ┆ ---      ┆ ---      ┆ ---                            │
│ i64      ┆ null     ┆ null     ┆ null     ┆ null     ┆ null     ┆ null                           │
╞══════════╪══════════╪══════════╪══════════╪══════════╪══════════╪════════════════════════════════╡
│ 11974053 ┆ null     ┆ null     ┆ null     ┆ null     ┆ null     ┆ null                           │
└──────────┴──────────┴──────────┴──────────┴──────────┴──────────┴────────────────────────────────┘

Schema:
Schema({'match_id': Int64, 'period_1': Null, 'period_2': Null, 'period_3': Null, 'period_4': Null, 'period_5': Null, 'current_period_start_timestamp': Null})


In [8]:
# 5. Check schemas of all Time files


schemas = Counter()


# Read every Time parquet file and store its schema

for file in time_files:

    df = pl.read_parquet(file)

    schema_tuple = tuple(
        df.schema.items()
    )

    schemas[schema_tuple] += 1


print("Number of different schemas:",len(schemas))

Number of different schemas: 5


In [9]:
# 6. Display all different Time schemas


for i, (schema, count) in enumerate(schemas.items(), start=1):

    print("=" * 60)

    print(f"Schema {i}")
    print(f"Number of files: {count}")

    print("-" * 60)

    for column, dtype in schema:
        print(f"{column} -> {dtype}")

Schema 1
Number of files: 12545
------------------------------------------------------------
match_id -> Int64
period_1 -> Null
period_2 -> Null
period_3 -> Null
period_4 -> Null
period_5 -> Null
current_period_start_timestamp -> Null
Schema 2
Number of files: 6886
------------------------------------------------------------
match_id -> Int64
period_1 -> Int64
period_2 -> Int64
period_3 -> Int64
period_4 -> Null
period_5 -> Null
current_period_start_timestamp -> Int64
Schema 3
Number of files: 15585
------------------------------------------------------------
match_id -> Int64
period_1 -> Int64
period_2 -> Int64
period_3 -> Null
period_4 -> Null
period_5 -> Null
current_period_start_timestamp -> Int64
Schema 4
Number of files: 550
------------------------------------------------------------
match_id -> Int64
period_1 -> Null
period_2 -> Null
period_3 -> Null
period_4 -> Null
period_5 -> Null
current_period_start_timestamp -> Int64
Schema 5
Number of files: 105
-------------------------

# Schema Comparison

Before combining all Time parquet files, we checked the schema of each file to identify possible structural inconsistencies.

Although all Time files contain the same columns, different files may have different data types because some columns contain only missing values in certain snapshots. For example, a period column can be stored as Null when no time information is available, while other files store the same column as Int64.

The purpose of displaying the schema summary is to:

- Verify that all files have the same column structure.
- Identify differences in data types between parquet files.
- Understand whether inconsistencies are caused by missing values or by real structural differences.
- Define a unified schema before concatenating all files.

After checking the schema summary, we confirmed that all files have the same set of columns, and the differences are only related to data types caused by missing values.
Therefore, a standard schema can be applied before concatenation while preserving null values.

In [10]:
#7. Display schema summary


for i, (schema, count) in enumerate(schemas.items(), start=1):

    print(f"\nSchema {i} - Files: {count}")

    print(
        [column for column, dtype in schema]
    )


Schema 1 - Files: 12545
['match_id', 'period_1', 'period_2', 'period_3', 'period_4', 'period_5', 'current_period_start_timestamp']

Schema 2 - Files: 6886
['match_id', 'period_1', 'period_2', 'period_3', 'period_4', 'period_5', 'current_period_start_timestamp']

Schema 3 - Files: 15585
['match_id', 'period_1', 'period_2', 'period_3', 'period_4', 'period_5', 'current_period_start_timestamp']

Schema 4 - Files: 550
['match_id', 'period_1', 'period_2', 'period_3', 'period_4', 'period_5', 'current_period_start_timestamp']

Schema 5 - Files: 105
['match_id', 'period_1', 'period_2', 'period_3', 'period_4', 'period_5', 'current_period_start_timestamp']


In [11]:
# 8.Define standard schema for Time dataset


time_schema = {
    "match_id": pl.Int64,
    "period_1": pl.Int64,
    "period_2": pl.Int64,
    "period_3": pl.Int64,
    "period_4": pl.Int64,
    "period_5": pl.Int64,
    "current_period_start_timestamp": pl.Int64
}

print(time_schema)

{'match_id': Int64, 'period_1': Int64, 'period_2': Int64, 'period_3': Int64, 'period_4': Int64, 'period_5': Int64, 'current_period_start_timestamp': Int64}


In [12]:
#9. Read and process all Time files


# Store processed Time DataFrames

time_frames = []


# Process every Time parquet file

for file in time_files:

    df = pl.read_parquet(file)

    snapshot_date = file.parent.name

    df = df.with_columns(
        pl.lit(snapshot_date)
        .str.strptime(
            pl.Date,
            "%Y%m%d"
        )
        .alias("snapshot_date")
    )



    df = df.cast(
        time_schema,
        strict=False
    )



    time_frames.append(df)



print("Number of processed Time files:",len(time_frames))

Number of processed Time files: 35671


In [15]:
# 10. Concatenate all Time DataFrames

# Combine all processed Time DataFrames vertically.
# Each row represents a Time record from a snapshot.

time_snapshot = pl.concat(
    time_frames,
    how="vertical"
)


print("Final Time dataset shape:",time_snapshot.shape)

Final Time dataset shape: (35671, 8)


In [16]:
# 11. Check missing values in Time dataset

# Count missing values in each column

print(
    time_snapshot.null_count()
)

shape: (1, 8)
┌──────────┬──────────┬──────────┬──────────┬──────────┬──────────┬────────────────┬───────────────┐
│ match_id ┆ period_1 ┆ period_2 ┆ period_3 ┆ period_4 ┆ period_5 ┆ current_period ┆ snapshot_date │
│ ---      ┆ ---      ┆ ---      ┆ ---      ┆ ---      ┆ ---      ┆ _start_timesta ┆ ---           │
│ u32      ┆ u32      ┆ u32      ┆ u32      ┆ u32      ┆ u32      ┆ mp             ┆ u32           │
│          ┆          ┆          ┆          ┆          ┆          ┆ ---            ┆               │
│          ┆          ┆          ┆          ┆          ┆          ┆ u32            ┆               │
╞══════════╪══════════╪══════════╪══════════╪══════════╪══════════╪════════════════╪═══════════════╡
│ 0        ┆ 13095    ┆ 13200    ┆ 28785    ┆ 35671    ┆ 35671    ┆ 12545          ┆ 0             │
└──────────┴──────────┴──────────┴──────────┴──────────┴──────────┴────────────────┴───────────────┘


In [17]:
# 12. Check duplicated rows

duplicate_count = (
    time_snapshot
    .is_duplicated()
    .sum()
)


print("Number of duplicated rows:",duplicate_count)

Number of duplicated rows: 0


In [19]:
# 13. Check match IDs with multiple snapshots


time_snapshot_counts = (
    time_snapshot
    .group_by("match_id")
    .agg(
        pl.col("snapshot_date")
        .n_unique()
        .alias("number_of_snapshots")
    )
)


# Find matches appearing in more than one snapshot

multiple_time_snapshots = (
    time_snapshot_counts
    .filter(
        pl.col("number_of_snapshots") > 1
    )
)


print("Match IDs with multiple snapshots:",multiple_time_snapshots.shape[0])


multiple_time_snapshots.head(10)

Match IDs with multiple snapshots: 16343


match_id,number_of_snapshots
i64,u32
12110649,2
12156487,2
12174324,2
12189684,2
12157434,2
12165702,2
12060306,2
12099243,2
12041814,2


In [20]:
# 14. Check if Time information changes between snapshots

time_changes = (
    time_snapshot
    .group_by("match_id")
    .agg(
        pl.col("period_1")
        .n_unique()
        .alias("different_period_1"),

        pl.col("period_2")
        .n_unique()
        .alias("different_period_2"),

        pl.col("period_3")
        .n_unique()
        .alias("different_period_3"),

        pl.col("period_4")
        .n_unique()
        .alias("different_period_4"),

        pl.col("period_5")
        .n_unique()
        .alias("different_period_5"),

        pl.col("current_period_start_timestamp")
        .n_unique()
        .alias("different_timestamps")
    )
)


# Find matches where Time data changed

changed_time_matches = (
    time_changes
    .filter(
        (pl.col("different_period_1") > 1)
        |
        (pl.col("different_period_2") > 1)
        |
        (pl.col("different_period_3") > 1)
        |
        (pl.col("different_period_4") > 1)
        |
        (pl.col("different_period_5") > 1)
        |
        (pl.col("different_timestamps") > 1)
    )
)


print("Number of matches with Time changes:",changed_time_matches.shape[0])


changed_time_matches.head(10)


Number of matches with Time changes: 1939


match_id,different_period_1,different_period_2,different_period_3,different_period_4,different_period_5,different_timestamps
i64,u32,u32,u32,u32,u32,u32
12086533,2,2,1,1,1,2
12148633,1,1,2,1,1,1
12086241,1,1,1,1,1,2
12118220,2,2,2,1,1,1
12109973,1,2,1,1,1,1
12107743,2,2,1,1,1,2
12104727,1,2,1,1,1,1
12123889,1,2,1,1,1,1
12079578,1,2,1,1,1,1


In [21]:
# 15. Check matches with multiple snapshots and time changes


print("Total rows:",time_snapshot.shape[0])


print("Total unique matches:",time_snapshot["match_id"].n_unique())


print("Matches with time changes:",changed_time_matches.shape[0])

Total rows: 35671
Total unique matches: 16873
Matches with time changes: 1939


In [22]:
# 16. Check match_id consistency

time_consistency = (
    time_snapshot
    .group_by("match_id")
    .agg(
        pl.col("period_1").n_unique().alias("unique_period_1"),
        pl.col("period_2").n_unique().alias("unique_period_2"),
        pl.col("current_period_start_timestamp")
        .n_unique()
        .alias("unique_timestamps")
    )
)

print(time_consistency.head())

shape: (5, 4)
┌──────────┬─────────────────┬─────────────────┬───────────────────┐
│ match_id ┆ unique_period_1 ┆ unique_period_2 ┆ unique_timestamps │
│ ---      ┆ ---             ┆ ---             ┆ ---               │
│ i64      ┆ u32             ┆ u32             ┆ u32               │
╞══════════╪═════════════════╪═════════════════╪═══════════════════╡
│ 12199625 ┆ 1               ┆ 1               ┆ 1                 │
│ 12177611 ┆ 1               ┆ 1               ┆ 1                 │
│ 12060693 ┆ 1               ┆ 1               ┆ 1                 │
│ 12188240 ┆ 1               ┆ 1               ┆ 1                 │
│ 12171168 ┆ 1               ┆ 1               ┆ 1                 │
└──────────┴─────────────────┴─────────────────┴───────────────────┘


In [23]:
# 17. Count matches with inconsistent time values


inconsistent_time = (
    time_consistency
    .filter(
        (pl.col("unique_period_1") > 1)
        |
        (pl.col("unique_period_2") > 1)
        |
        (pl.col("unique_timestamps") > 1)
    )
)


print("Matches with inconsistent Time values:",inconsistent_time.shape[0])


inconsistent_time.head(10)

Matches with inconsistent Time values: 1878


match_id,unique_period_1,unique_period_2,unique_timestamps
i64,u32,u32,u32
12121111,2,2,1
12086238,2,2,2
12099609,2,2,2
12079587,2,2,2
12148639,1,2,1
12118601,1,2,1
12144179,2,1,1
12124803,1,2,1
12144191,1,2,1


In [24]:
# 18. Save cleaned Time dataset


clean_path = DATA_ROOT / "Data"

clean_path.mkdir(
    parents=True,
    exist_ok=True
)


time_snapshot.write_parquet(
    clean_path / "time_clean.parquet"
)


print("Saved successfully:")

print(clean_path / "time_clean.parquet")

Saved successfully:
/Users/macbook/Desktop/ data_analysis_tannis projects/tennis_data_analysis/tennis_data/Data/time_clean.parquet


In [27]:
#19
time_clean = pl.read_parquet(
    clean_path / "time_clean.parquet"
)

print(time_clean.shape)
print(time_clean.schema)

time_clean.head(20)

(35671, 8)
Schema({'match_id': Int64, 'period_1': Int64, 'period_2': Int64, 'period_3': Int64, 'period_4': Int64, 'period_5': Int64, 'current_period_start_timestamp': Int64, 'snapshot_date': Date})


match_id,period_1,period_2,period_3,period_4,period_5,current_period_start_timestamp,snapshot_date
i64,i64,i64,i64,i64,i64,i64,date
11974053,null,null,null,null,null,null,2024-02-01
11974066,null,null,null,null,null,null,2024-02-01
11998445,3259,2639,4202,null,null,1706816851,2024-02-01
11998446,2488,2375,null,null,null,1706803981,2024-02-01
11998447,3741,1913,null,null,null,1706798459,2024-02-01
…,…,…,…,…,…,…,…
11998672,4604,5172,3700,null,null,1706751953,2024-02-01
11998674,3359,7619,3338,null,null,1706762236,2024-02-01
11998675,3446,3268,4263,null,null,1706749125,2024-02-01
